# MAX IV: separate summed helicities using a saturated reference

State **0** is the saturated pair **+1136 / −1134**, with darks **1133, 1135, 1137**, acquired at **25 × 0.6 s**. State **1** is the magnetic pair **+1141 / −1139**, with darks **1138, 1140, 1142**, acquired at **25 × 1.2 s**. Each pair is dark-corrected and exposure-normalized before its **intensities** are summed. Only those two sums enter retrieval; measured individual helicities remain held out for validation.

Use `MODES=[1,1]` or `[1,1,2]`: the first two modes represent opposite helicities and both enter the joint physical model

`log psi(s, ±, r) = C(r) + t(r) [q_charge ± q_magnetic m_s(r)]`.

Illumination, relative thickness and complex response are shared between states. State 0 has uniform `m_0=SATURATED_SIGN` inside material; state 1 has a fitted real magnetization map. The default sign −1 follows the negative saturated field metadata and is a convention unless the magnetic response sign is independently calibrated. Saturation means no magnetic **domain variation**, not zero magnetic transmission. This constraint supplies a known spatial state and magnetization scale, but does not restore a directly measured heterodyne term or guarantee unique separation.

This experiment uses two magnetic states at essentially one energy. It complements the multi-energy notebook. The largest support aperture is assumed to contain magnetic material with unit relative thickness; smaller reference holes are treated as nonmagnetic. Inspect the aperture and shared-illumination assumption before interpreting the result.


The solver alternates the existing phase-retrieval kernel and physical projector. After warmup, every round performs **joint physical projection of both candidate helicities → phase retrieval against each measured sum**, then repeats using the resulting fields. `MODES=[1,1,2]` adds a nonmagnetic background mode; only the first two modes enter the helicity fit.

`ITERATIONS` controls the initial warmup, `INNER_ITERATIONS` controls phase-retrieval steps between projections, `OUTER_ITERATIONS` controls the number of cycles, and `PROJECTION_RELAXATION` controls the physical update strength. The default warmup is shortened to HAPRE 100 + ER 30, followed by 30 physical/ER cycles with 10 ER iterations each. These are tunable schedules, not established convergence settings.

Keep separate diagnostics after physical projection and after phase retrieval: enforcing the summed intensity does not establish helicity separation. Individual helicities and their difference remain held out. Restart the kernel after changing the shared libraries.


In [ ]:
%matplotlib inline
from pathlib import Path
import sys, json, re
import h5py
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import set_workers
ROOT = Path.cwd()
if (ROOT / "maxiv_phase_test").is_dir(): ROOT = ROOT / "maxiv_phase_test"
REPO = ROOT.parent
sys.path.insert(0, str(REPO / "library"))
import CCI_core as cci
import fth_phase_workflow as wf
import phase_retrieval_universal as pr
import phase_retrieval_universal as universal
import phase_retrieval_geometry as geometry
from retrieval_progress import retrieval_progress

DATA_ROOT = Path("/media/riccardo/EXT_16TB1/DATA/MAXIV/MaxIV2409")
RAW_FOLDER, AVERAGE_FOLDER = DATA_ROOT / "raw", DATA_ROOT / "proc"
SUPPORT_FILE = ROOT / "processed/Logs/data_recon_ImId_1143_maxiv.hdf5"
OUTPUT_FILE = ROOT / "processed/Logs/data_phase_summed_helicities_saturated_maxiv.hdf5"
STATE_PAIRS = [
    dict(state=0, positive=1136, negative=1134, darks=[1133,1135,1137], frames=25, exposure_s=0.6),
    dict(state=1, positive=1141, negative=1139, darks=[1138,1140,1142], frames=25, exposure_s=1.2),
]
# Existing load_cmos describes scan_*_avg.h5 as averaged frames.
CACHE_REDUCTION = "mean"  # "sum" divides each scan by its metadata frame count first.
DETECTOR_CENTER = (1137, 1126)
CROP = 248
BINNING = 2
SATURATION = 0.75e4  # Per-frame mean-count threshold, before exposure normalization.
USE_SAVED_PIXEL_MASK = False
BEAMSTOP_RADIUS = 0
ENERGY_TOLERANCE_EV = 0.25
FIELD_TOLERANCE = 0.005
MODES = [1, 1]  # Or [1, 1, 2]: +, -, nonmagnetic background.
ALGORITHMS = ["HAPRE", "ER"]
ITERATIONS = [100, 30]  # Warmup; increase if needed.
OUTER_ITERATIONS = 30
INNER_ITERATIONS = 10
INITIALIZATION_SEED = 0
FFT_WORKERS = 1
USE_PHYSICAL_PROJECTION = True
USE_SATURATION_CONSTRAINT = True  # False: control run with the same input sums but no uniform-state constraint.
SATURATED_SIGN = -1
COMMON_BACKGROUND = True
PROJECTION_RELAXATION = 0.3
FOCUS_PROP_UM = 0.0  # Adjust if sample-plane focusing is needed.
FOCUS_PHASE_RAD = 0.0
FOCUS_SETUP = {"ccd_dist": 0.195, "px_size": 11e-6 * BINNING}


## Validate metadata, normalize exposures, then sum
The raw scan title and CMOS list must match the specified frame count and exposure. Each state's darks must have the same exposure and magnetic field as its two helicities. Small energy drifts below `ENERGY_TOLERANCE_EV` are treated as a single energy; no energy resampling is applied here.

With `CACHE_REDUCTION="mean"`, corrected intensity is `(mean_signal − mean_of_dark_means) / exposure_s`. The 25 frames improve statistics and are **not** divided out again. With `"sum"`, every cached scan is first divided by its own verified frame count. There is no inherited `FRAME_NORMALIZATION=500` factor. Scalar incident flux is assumed stable; no fitted per-helicity scale is used to erase dichroism. Saturation detection on cached averages cannot detect isolated saturated frames that were averaged below threshold.

Apply the same detector centering, crop and binning to all four images. Preserve state 0 as a summed target, without subtracting it from state 1 or freezing its arbitrary warmup mode decomposition.


In [ ]:
def scan_snapshot(scan_id):
    path = RAW_FOLDER / f"2409_maxiv_{scan_id:05d}.h5"
    with h5py.File(path,"r") as h:
        entry = h[f"entry{scan_id}"]
        snapshot = entry["measurement/pre_scan_snapshot"]
        title = entry["title"][()]
        if isinstance(title,bytes): title = title.decode()
        match = re.fullmatch(r"timescan\s+(\d+)\s+([\d.eE+-]+)\s+([\d.eE+-]+)",str(title).strip())
        if match is None: raise ValueError(f"Scan {scan_id}: cannot validate acquisition from {title!r}")
        frames, exposure = int(match[1]), float(match[2])
        if len(entry["measurement/cmos"]) != frames:
            raise ValueError(f"Scan {scan_id}: frame-list/title mismatch")
        def scalar(name):
            values = np.asarray(snapshot[name][()]).reshape(-1).astype(float)
            if not np.all(np.isfinite(values)) or not np.allclose(values,values[0]):
                raise ValueError(f"Scan {scan_id}: invalid/varying {name}")
            return float(values[0])
        return dict(scan=scan_id,frames=frames,exposure_s=exposure,energy=scalar("energy"),
                    field=scalar("magOOP"),undulator_phase=scalar("unduphase"))

def validate_manifest(rows):
    ids = [row["state"] for row in rows]
    if len(ids)<2 or len(ids)!=len(set(ids)) or ids.count(0)!=1:
        raise ValueError("Require state 0 and at least one other unique magnetic state")
    scans = [r[k] for r in rows for k in ("positive","negative")]
    if len(scans)!=len(set(scans)): raise ValueError("Do not reuse a helicity measurement")
    if set(scans) & {s for r in rows for s in r["darks"]}:
        raise ValueError("A signal scan cannot also be a dark")
    for row in rows:
        if row["frames"] <= 0 or not np.isfinite(row["exposure_s"]) or row["exposure_s"] <= 0 or not row["darks"]:
            raise ValueError("Require positive frame count/exposure and dark references")
    return sorted(rows,key=lambda row:(row["state"]!=0,row["state"]))

STATE_PAIRS = validate_manifest(STATE_PAIRS)
if CACHE_REDUCTION not in {"mean","sum"}: raise ValueError("CACHE_REDUCTION must be mean or sum")
if SATURATED_SIGN not in (-1,1): raise ValueError("SATURATED_SIGN must be +/-1")
metadata = []
scan_metadata = {}
for row in STATE_PAIRS:
    ids = [row["positive"],row["negative"],*row["darks"]]
    for scan in ids:
        meta = scan_snapshot(scan)
        if meta["frames"] != row["frames"] or not np.isclose(meta["exposure_s"],row["exposure_s"]):
            raise ValueError(f"Scan {scan}: acquisition differs from specified frames/exposure")
        scan_metadata[scan] = meta
    if np.ptp([scan_metadata[s]["field"] for s in ids]) > FIELD_TOLERANCE:
        raise ValueError(f"State {row['state']}: fields differ within the pair/darks")
    metadata.append([scan_metadata[row["positive"]],scan_metadata[row["negative"]]])
all_energies = np.asarray([m["energy"] for m in scan_metadata.values()])
if np.min(all_energies)<=0 or np.ptp(all_energies)>ENERGY_TOLERANCE_EV:
    raise ValueError("Use one energy across states in this notebook")
ENERGY_EV = float(np.mean([m["energy"] for pair in metadata for m in pair]))
state_labels = [f"state_{row['state']}" for row in STATE_PAIRS]
print("State order:",state_labels,"energy:",ENERGY_EV)
print("Fields (+,-):",[[m["field"] for m in pair] for pair in metadata])
print("Undulator phase (+,-):",[[m["undulator_phase"] for m in pair] for pair in metadata])

saved = wf.load_data_dict(SUPPORT_FILE)
support_source = np.asarray(saved["supportmask"]) != 0
saved_mask = np.asarray(saved["mask_pixel"]) != 0
if support_source.ndim!=2 or saved_mask.shape!=support_source.shape:
    raise ValueError("Support and saved mask must be matching 2D arrays")
def load_mean(scan):
    with h5py.File(AVERAGE_FOLDER/f"scan_{scan:04d}_avg.h5","r") as h:
        image = np.asarray(h["image"][()]).squeeze().astype(float)
    if image.shape!=support_source.shape: raise ValueError("Input/support shape mismatch")
    if CACHE_REDUCTION=="sum": image /= scan_metadata[scan]["frames"]
    return image

images,masks = [],[]
for row in STATE_PAIRS:
    dark = np.mean([load_mean(scan) for scan in row["darks"]],axis=0)
    for scan in (row["positive"],row["negative"]):
        raw = load_mean(scan)
        image = wf.center_image(raw-dark,DETECTOR_CENTER,cci)/row["exposure_s"]
        mask = wf.center_image(((raw>=SATURATION)|~np.isfinite(raw)|~np.isfinite(dark)).astype(np.uint8),DETECTOR_CENTER,cci)!=0
        mask |= ~np.isfinite(image)
        if USE_SAVED_PIXEL_MASK: mask |= saved_mask
        if BEAMSTOP_RADIUS>0:
            y,x=np.indices(image.shape)
            mask |= np.hypot(y-image.shape[0]//2,x-image.shape[1]//2)<BEAMSTOP_RADIUS
        images.append(np.nan_to_num(image).astype(np.float32)); masks.append(mask)
prepared,prepared_masks,support,_,input_geometry = geometry.prepare(
    np.stack(images),np.stack(masks),support_source,
    dict(crop=CROP,binning=BINNING,roi=None,Nmodes=1))
prepared = prepared.reshape(len(STATE_PAIRS),2,*support.shape)
prepared_masks = prepared_masks.reshape(prepared.shape)
summed_intensity = prepared.sum(axis=1)
mask_pixel = np.any(prepared_masks!=0,axis=1)
mask_pixel |= ~np.isfinite(summed_intensity)|(summed_intensity<0)
del images,masks
modal_supports = pr.mode_supports(support,MODES,support.shape)
material_thickness = universal.largest_support_component(support).astype(float)
fig,axes=plt.subplots(1,4,figsize=(15,4))
for ax,img,title in zip(axes,[*np.log1p(np.maximum(summed_intensity,0)),support,material_thickness],
                       ["State 0: saturated sum","State 1: magnetic sum","Support","Assumed material aperture"]):
    ax.imshow(img); ax.set_title(title)
plt.show()


## Alternate physical projection and phase retrieval
Warm up each summed observation using the configured HAPRE/ER schedule. Each outer round then:

1. Fits the existing shared charge/magnetic model to **both** candidate helicity fields across all observations, applying the relaxed physical projection.
2. Uses those projected fields as the starts for the next ER phase-retrieval block against each observation's **summed** intensity and modal supports.
3. Carries those updated fields into the next physical projection.

The first two modes are flattened into coherent +/− observations only while applying `project_fourier_fields_general`; the optional background never enters that magnetic fit. The final returned fields are the result of phase retrieval following the last physical projection. The saved component maps describe the last physical fit, not an exact model representation of the subsequent phase-retrieval fields. Both stages' summed-data residuals are saved.


In [ ]:
if list(MODES) not in ([1, 1], [1, 1, 2]):
    raise ValueError("Choose MODES=[1,1] or [1,1,2]; first two modes are + and - helicity")
if OUTER_ITERATIONS < 1 or INNER_ITERATIONS < 1:
    raise ValueError("Iteration counts must be positive")
nmodes = len(MODES)
mode_labels = ["Candidate +", "Candidate -"] + (["Nonmagnetic background"] if nmodes == 3 else [])
assert modal_supports.shape == (nmodes, *support.shape)
rng = np.random.default_rng(INITIALIZATION_SEED)
def detector_project(fields, intensity, excluded):
    power = np.sum(abs(fields)**2, axis=0)
    valid = ~excluded
    projected = fields.copy()
    nonzero = valid & (power > 0)
    projected[:, nonzero] *= np.sqrt(intensity[nonzero]/power[nonzero])
    zero = valid & (power == 0)
    projected[:, zero] = np.sqrt(intensity[zero]/fields.shape[0])
    return projected

initial_fields = []
for target, excluded in zip(summed_intensity, mask_pixel):
    objects = modal_supports * np.exp(1j*rng.uniform(-np.pi, np.pi, (nmodes, *support.shape)))
    f = np.fft.ifftshift(np.fft.ifft2(np.fft.fftshift(objects, axes=(-2,-1))), axes=(-2,-1))
    valid = (~excluded) & (target > 0)
    if not np.any(valid): raise ValueError("No positive usable data")
    f *= np.sqrt(target[valid].sum()/np.sum(abs(f[:,valid])**2))
    initial_fields.append(f)
initial_fields = np.stack(initial_fields)
nstages = len(ALGORITHMS)
recipe = pr.default_phase_retrieval_recipe()
recipe.update(
    modes=MODES, algorithm_list=ALGORITHMS, number_iterations=ITERATIONS,
    helicity=["sum"]*nstages, beta_zero=[.5]*nstages,
    beta_mode=["arctan" if mode=="HAPRE" else "const" for mode in ALGORITHMS],
    alpha_zero=[0.]*nstages, alpha_mode=["const"]*nstages,
    RL_its=[0]*nstages, RL_freqs=[10**9]*nstages, TV_freqs=[10**9]*nstages,
    plot_every=[10**9]*nstages, average_img=[1]*nstages,
    Fourier_last=[True]*nstages, output=[True]*nstages,
    Startimage=[initial_fields[0]]+["sum"]*(nstages-1), Startgamma=[None]*nstages,
    crop=0, binning=1, roi=None, hologram_intensity_cutoff_vmin=-1,
    return_format="dict",
)

def update_pair(f, index, warmup=False):
    local = dict(recipe)
    if warmup:
        local["Startimage"] = [f]+["sum"]*(nstages-1)
    else:
        # Build a single ER stage from the existing validated recipe.
        local = {key: ([value[-1]] if isinstance(value, list) and key != "modes" else value)
                 for key, value in recipe.items()}
        local.update(algorithm_list=["ER"], number_iterations=[INNER_ITERATIONS],
                     Startimage=[f], helicity=["sum"], beta_mode=["const"])
    with retrieval_progress(local, kind="unified", leave=False,
                            description=state_labels[index]):
        result = pr.phase_retrieval_algorithm({"sum": summed_intensity[index]},
            mask_pixel=mask_pixel[index], supportmask=modal_supports, phase_retrieval_recipe=local)
    return result["full_coherence"]["sum"]

physical_recipe = universal.default_general_phase_retrieval_recipe()
physical_recipe.update(physical_phase_reference=True, energy_values=np.array([ENERGY_EV]),
    projection_focus_prop_um=FOCUS_PROP_UM, projection_focus_phase_rad=FOCUS_PHASE_RAD,
    projection_focus_setup={**FOCUS_SETUP, "energy": ENERGY_EV})
flat_energies = [ENERGY_EV] * (2*len(state_labels))
flat_polarizations = np.tile([1., -1.], len(state_labels)).tolist()
labels = np.repeat(state_labels, 2).tolist()
beams = ["beam0"] * len(labels)
saturated_states = {"state_0": SATURATED_SIGN} if USE_SATURATION_CONSTRAINT else None

def physical_project(stack):
    projected, components = universal.project_fourier_fields_general(
        stack[:, :2].reshape(-1, *support.shape), labels, flat_energies, flat_polarizations, beams,
        projection_model="physical_factorized", relaxation=PROJECTION_RELAXATION,
        recipe=physical_recipe, physical_iterations=20, saturated_states=saturated_states,
        material_thickness=np.fft.fftshift(material_thickness),
        projection_supportmask=np.fft.fftshift(support) != 0,
        magnetization_supportmask=np.fft.fftshift(material_thickness) != 0,
        return_components=True)
    output = stack.copy()
    output[:, :2] = projected.reshape(stack.shape[0], 2, *support.shape)
    if stack.shape[1] == 3 and COMMON_BACKGROUND:
        background = stack[:,2].copy()
        reference = background[0]
        for i in range(1,len(background)):
            overlap = np.vdot(reference, background[i])
            if abs(overlap) > 0: background[i] *= np.exp(-1j*np.angle(overlap))
        output[:,2] = background.mean(axis=0)
    return output, components

def sum_errors(stack):
    prediction = np.sum(abs(stack)**2, axis=1)
    return np.asarray([np.linalg.norm((pred-target)[~mask])/max(np.linalg.norm(target[~mask]),1e-30)
                       for pred,target,mask in zip(prediction,summed_intensity,mask_pixel)])

history, retrieval_history, components = [], [], None
with set_workers(FFT_WORKERS):
    fields = np.stack([update_pair(f, i, warmup=True) for i,f in enumerate(initial_fields)])
    warmup_fields = fields.copy()
    for outer in range(OUTER_ITERATIONS):
        if USE_PHYSICAL_PROJECTION:
            fields, components = physical_project(fields)
        # Preserve the last physical-stage fields separately from the subsequent PR.
        physical_fields = fields.copy()
        history.append(sum_errors(physical_fields))
        fields = np.stack([update_pair(f, i) for i,f in enumerate(physical_fields)])
        retrieval_history.append(sum_errors(fields))
        print(f"Round {outer+1}/{OUTER_ITERATIONS}: physical residual {np.mean(history[-1]):.5g}, "
              f"after phase retrieval {np.mean(retrieval_history[-1]):.5g}",flush=True)
pre_detector_fields = physical_fields  # Compatibility name for last physical-stage output.


## Validate separation after the alternating cycles
Compare the original, held-out helicity difference to the predicted difference, reporting both global assignments without per-energy swaps or rescaling. Identical candidate helicities predict zero difference and give relative difference error **1**. Plot the errors after physical projection and after phase retrieval separately; a low final sum residual can simply reflect the Fourier constraint.

Component maps belong to the last physical fit. Final fields belong to the following phase-retrieval block. The original measured individual helicities are never separate retrieval targets. With three modes, validation assumes half the background intensity in each helicity; that equal allocation cancels from their difference.


In [ ]:
prediction = np.sum(abs(fields)**2, axis=1)
modal_intensities = abs(fields)**2
print("Last physical-stage sum errors:", sum_errors(physical_fields))
print("Final phase-retrieval sum errors:", sum_errors(fields))
background_intensity = (modal_intensities[:, 2] if nmodes == 3 else np.zeros_like(prediction))
helicity_intensities = modal_intensities[:, :2] + background_intensity[:, None]/2
print("Modal power fractions per state:", modal_intensities.sum(axis=(-2,-1))/np.maximum(prediction.sum(axis=(-2,-1))[:,None],1e-30))
heldout_errors = np.empty((2, len(state_labels)))
for swap in (0,1):
    candidates = helicity_intensities[:, ::-1] if swap else helicity_intensities
    for i in range(len(state_labels)):
        valid = np.broadcast_to(~mask_pixel[i], prepared[i].shape)
        heldout_errors[swap,i] = np.linalg.norm((candidates[i]-prepared[i])[valid])/max(np.linalg.norm(prepared[i][valid]),1e-30)
print("Held-out errors (direct, globally swapped):", heldout_errors)
print("Assignment remains unanchored during retrieval; diagnostic best global swap:", int(np.argmin(np.mean(heldout_errors,axis=1))))
predicted_difference = helicity_intensities[:,0] - helicity_intensities[:,1]
measured_difference = prepared[:,0] - prepared[:,1]
difference_errors = np.asarray([
    [np.linalg.norm((sign*pred-obs)[~mask])/max(np.linalg.norm(obs[~mask]),1e-30)
     for pred,obs,mask in zip(predicted_difference,measured_difference,mask_pixel)]
    for sign in (1,-1)])
print("Held-out helicity-difference errors (direct, global swap):", difference_errors)
focused_fields = universal._projection_focus_transform(
    fields.reshape(-1,*support.shape), [ENERGY_EV]*(len(state_labels)*nmodes),
    FOCUS_PROP_UM, FOCUS_PHASE_RAD, {**FOCUS_SETUP,"energy":ENERGY_EV})
focused_fields = focused_fields.reshape(fields.shape)
exit_waves = np.fft.ifftshift(np.fft.fft2(np.fft.fftshift(focused_fields,axes=(-2,-1))),axes=(-2,-1))
roi = geometry.support_bounding_roi(support)
DISPLAY_STATE = min(1,len(state_labels)-1)
fig, axes = plt.subplots(nmodes,3,figsize=(12,3.5*nmodes))
for mode in range(nmodes):
    axes[mode,0].imshow(abs(exit_waves[DISPLAY_STATE,mode])[roi],cmap="gray")
    axes[mode,1].imshow(np.angle(exit_waves[DISPLAY_STATE,mode])[roi],cmap="twilight",vmin=-np.pi,vmax=np.pi)
    axes[mode,2].imshow(np.log1p(modal_intensities[DISPLAY_STATE,mode]),cmap="magma")
    for ax,title in zip(axes[mode],["Object amplitude", "Object phase", "Predicted intensity"]):
        ax.set_title(f"{mode_labels[mode]}: {title}")
fig.tight_layout()
fig, axes = plt.subplots(1,2,figsize=(11,4))
axes[0].semilogy(np.arange(1,len(history)+1), np.maximum(np.mean(history,axis=1),1e-16),label="After physical projection")
axes[0].semilogy(np.arange(1,len(retrieval_history)+1), np.maximum(np.mean(retrieval_history,axis=1),1e-16),label="After phase retrieval")
axes[0].legend()
axes[0].set(xlabel="Physical / phase-retrieval cycle", ylabel="Mean summed-intensity relative error")
for swap in (0,1): axes[1].plot(np.arange(len(state_labels)),difference_errors[swap],"o",label=["Direct","Global swap"][swap])
axes[1].set(xlabel="State index in selected stack",ylabel="Held-out difference relative error")
axes[1].axhline(1.,color="black",linestyle="--",label="No separation")
axes[1].legend()
plt.show()

if components is not None:
    magnetization = np.asarray(components["magnetization"])
    fig, axes = plt.subplots(1,len(state_labels),figsize=(4*len(state_labels),4),squeeze=False)
    for i,label in enumerate(components["state_names"]):
        axes[0,i].imshow(np.fft.ifftshift(magnetization[i])[roi],cmap="RdBu_r",vmin=-1,vmax=1)
        axes[0,i].set_title(label + (" (fixed uniform)" if label == "state_0" and USE_SATURATION_CONSTRAINT else ""))
    plt.show()

# Show the strongest measured contrast for inspection only, never for fitting/selection.
contrast_norm=np.array([np.linalg.norm(d[~mask]) for d,mask in zip(measured_difference,mask_pixel)])
inspect_index=int(np.argmax(contrast_norm))
valid=~mask_pixel[inspect_index]
obs=measured_difference[inspect_index]
pred=predicted_difference[inspect_index]
limit=np.percentile(abs(obs[valid]),99)
fig,axes=plt.subplots(1,3,figsize=(14,4))
for ax,img,title in zip(axes,[obs,pred,pred-obs],["Measured difference (held out)","Recovered difference (+/- convention)","Difference residual"]):
    ax.imshow(np.ma.masked_where(~valid,img),cmap="RdBu_r",vmin=-limit,vmax=limit);ax.set_title(title)
fig.tight_layout();plt.show()
print("Equal-helicity / zero-difference baseline error: 1.0")
print("Difference norm ratio:",np.linalg.norm(pred[valid])/max(np.linalg.norm(obs[valid]),1e-30))
print("Difference correlation:",np.corrcoef(obs[valid],pred[valid])[0,1])

valid_all=~mask_pixel
truth_norm=np.linalg.norm(measured_difference[valid_all])
global_difference_errors=np.array([
    np.linalg.norm((sign*predicted_difference-measured_difference)[valid_all])/max(truth_norm,1e-30)
    for sign in (1,-1)])
separation_summary=dict(global_difference_errors=global_difference_errors.tolist(),
    zero_difference_baseline=1.0,completed_cycles=OUTER_ITERATIONS,
    beats_zero_difference=bool(truth_norm>0 and np.min(global_difference_errors)<1.0))
print("Separation summary:",separation_summary)
if not separation_summary["beats_zero_difference"]:
    print("NO demonstrated separation: this run does not improve on predicting identical helicities.")
else:
    print("Difference prediction improves on equality; this alone does not prove unique recovery.")


In [ ]:
OUTPUT_FILE.parent.mkdir(parents=True,exist_ok=True)
with h5py.File(OUTPUT_FILE,"w") as out:
    out.attrs["workflow"] = "maxiv_summed_helicities_saturated_state0"
    out.attrs["state_pairs_json"] = json.dumps(STATE_PAIRS)
    out.attrs["metadata_json"] = json.dumps(scan_metadata)
    out.attrs["mode_labels_json"] = json.dumps(mode_labels)
    out.attrs["solver"]="alternating_physical_projection_then_phase_retrieval"
    out.attrs["separation_summary_json"]=json.dumps(separation_summary)
    out.attrs["settings_json"] = json.dumps(dict(modes=MODES, crop=CROP, binning=BINNING,
        energy_ev=ENERGY_EV, seed=INITIALIZATION_SEED, algorithms=ALGORITHMS, iterations=ITERATIONS,
        outer_iterations=OUTER_ITERATIONS, inner_iterations=INNER_ITERATIONS,
        use_physical_projection=USE_PHYSICAL_PROJECTION, projection_relaxation=PROJECTION_RELAXATION,
        saturation_constraint=USE_SATURATION_CONSTRAINT, saturated_sign=SATURATED_SIGN,
        common_background=COMMON_BACKGROUND, focus_prop_um=FOCUS_PROP_UM,
        focus_phase_rad=FOCUS_PHASE_RAD, focus_setup=FOCUS_SETUP,
        cache_reduction=CACHE_REDUCTION, detector_center=DETECTOR_CENTER, saturation=SATURATION,
        support_file=str(SUPPORT_FILE), use_saved_mask=USE_SAVED_PIXEL_MASK, beamstop_radius=BEAMSTOP_RADIUS))
    out.create_dataset("state_labels",data=np.asarray(state_labels,dtype="S"))
    for key,value in dict(fields=fields,initial_fields=initial_fields,warmup_fields=warmup_fields,
        pre_detector_fields=pre_detector_fields,exit_waves=exit_waves,
        measured_polarizations=prepared,summed_intensity=summed_intensity,
        mask_pixel=mask_pixel,supportmask=support,material_thickness=material_thickness,
        modal_supportmask=modal_supports,
        predicted_intensity=prediction,history=history,retrieval_history=retrieval_history,heldout_errors=heldout_errors,
        difference_errors=difference_errors,helicity_intensities=helicity_intensities,
        background_intensity=background_intensity,predicted_difference=predicted_difference).items():
        out.create_dataset(key,data=value,compression="gzip")
    if components is not None:
        group=out.create_group("last_physical_fit")
        group.attrs["note"]="Last physical fit before final phase-retrieval block; magnetization in shifted focused object convention"
        group.create_dataset("state_names",data=np.asarray(components["state_names"],dtype="S"))
        for key in ("common_log_objects","charge_response","magnetic_response","magnetization","material_thickness"):
            group.create_dataset(key,data=components[key],compression="gzip")
print("Saved:",OUTPUT_FILE)
